In [1]:
# importing libraries
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from pathlib import Path
from IPython.display import display

In [2]:
# data directory relative to project root
data_dir = Path("../data/world_dev_index")

# loading data from the World Development Index dataset
## consumption data (final consumption expenditure (constant 2015 US$))
cdata = pd.read_excel(
    data_dir / "cdata.xls",
    skiprows=3,
)

## real interest rate data (real interest rate (%))
rdata = pd.read_excel(
    data_dir / "rdata.xls",
    skiprows=3,
)
## changing data-format from wide to long
cdata = cdata.melt(
    id_vars= ["Country Name", "Country Code"],
    value_vars = [c for c in cdata.columns if str(c).isdigit()],
    var_name = "year",
    value_name = "consumption"
)
rdata = rdata.melt(
    id_vars= ["Country Name", "Country Code"],
    value_vars= [r for r in rdata.columns if str(r).isdigit()],
    var_name="year",
    value_name="interest_rate"
)

# adjoining cdata and rdata
df = pd.merge(
    cdata,
    rdata,
    on=["Country Code", "Country Name", "year"]
)

# sorting by country and year
df = df.sort_values(["Country Code", "year"])
df

,Country Name,Country Code,year,consumption,interest_rate
0,Aruba,ABW,1960,NaN,NaN
265,Aruba,ABW,1961,NaN,NaN
530,Aruba,ABW,1962,NaN,NaN
795,Aruba,ABW,1963,NaN,NaN
1060,Aruba,ABW,1964,NaN,NaN
...,...,...,...,...,...
16429,Zimbabwe,ZWE,2021,1.625061e+10,-30.318195
16694,Zimbabwe,ZWE,2022,1.812032e+10,-38.093651
16959,Zimbabwe,ZWE,2023,1.854998e+10,-68.877267
17224,Zimbabwe,ZWE,2024,1.884427e+10,-85.897032


In [3]:
df["interest_rate"] = df["interest_rate"]/100
df["c_growth"] = (
    df.groupby("Country Code")["consumption"].pct_change(periods=1, fill_method=None)
)

df["log_gross_c_growth"] = (
    np.log(1+ df["c_growth"])
)

df["r_lag1"] = (
    df.groupby("Country Code")["interest_rate"].shift(periods=1)
)

df["delta_r"] = (
    df.groupby("Country Code")["interest_rate"].diff(periods=1)
)

df["log_gross_r"] = (
    np.log(1+ df["interest_rate"])
)

df_cleaned = df.dropna()
df_cleaned

,Country Name,Country Code,year,consumption,interest_rate,c_growth,log_gross_c_growth,r_lag1,delta_r,log_gross_r
11399,Angola,AGO,2003,3.967169e+10,0.007748,0.048370,0.047236,-0.413295,0.421042,0.007718
11664,Angola,AGO,2004,4.017592e+10,0.367004,0.012710,0.012630,0.007748,0.359256,0.312621
11929,Angola,AGO,2005,4.448563e+10,0.196820,0.107271,0.101898,0.367004,-0.170184,0.179668
12194,Angola,AGO,2006,4.669973e+10,0.023083,0.049771,0.048572,0.196820,-0.173737,0.022820
12459,Angola,AGO,2007,5.037402e+10,0.119081,0.078679,0.075737,0.023083,0.095999,0.112508
...,...,...,...,...,...,...,...,...,...,...
16164,Zimbabwe,ZWE,2020,1.488635e+10,-0.806103,-0.081300,-0.084796,-0.766316,-0.039787,-1.640426
16429,Zimbabwe,ZWE,2021,1.625061e+10,-0.303182,0.091645,0.087685,-0.806103,0.502921,-0.361231
16694,Zimbabwe,ZWE,2022,1.812032e+10,-0.380937,0.115055,0.108904,-0.303182,-0.077755,-0.479547
16959,Zimbabwe,ZWE,2023,1.854998e+10,-0.688773,0.023711,0.023435,-0.380937,-0.307836,-1.167232


In [4]:
model1_all_bm = sm.OLS.from_formula(
    "c_growth ~ interest_rate",
    data = df_cleaned
).fit()

model1_all_logr = sm.OLS.from_formula(
    "c_growth ~ log_gross_r",
    data = df_cleaned
).fit()

model1_all_logr_logc = sm.OLS.from_formula(
    "log_gross_c_growth ~ log_gross_r",
    data = df_cleaned
).fit()

model1_vnm_bm = sm.OLS.from_formula(
    "c_growth ~ interest_rate",
   data=df_cleaned.query("`Country Code` == 'VNM'") 
).fit()

model1_vnm_logr = sm.OLS.from_formula(
    "c_growth ~ log_gross_r",
    data = df_cleaned.query("`Country Code` == 'VNM'") 
).fit()

model1_vnm_logr_logc = sm.OLS.from_formula(
    "log_gross_c_growth ~ log_gross_r",
    data = df_cleaned.query("`Country Code` == 'VNM'") 
).fit()

results_table_m1 = summary_col(
    [model1_all_bm, model1_all_logr, model1_all_logr_logc, model1_vnm_bm, model1_vnm_logr, model1_vnm_logr_logc],
    stars=True,
    model_names=["Model 1 (all, BM)", "Model 1 (all, with log(r))", "Model 1 (all, with log(r) and log(c))","Model 1 (VNM, BM)", "Model 1 (VNM, with log(r))", "Model 1 (VNM, with log(r) and log(c))"],
    info_dict={
        "N": lambda x: f"{int(x.nobs)}"
    },
)

print(results_table_m1)


               Model 1 (all, BM) Model 1 (all, with log(r)) Model 1 (all, with log(r) and log(c)) Model 1 (VNM, BM) Model 1 (VNM, with log(r)) Model 1 (VNM, with log(r) and log(c))
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Intercept      0.0390***         0.0385***                  0.0361***                             0.0621***         0.0615***                  0.0595***                            
               (0.0011)          (0.0010)                   (0.0010)                              (0.0045)          (0.0044)                   (0.0042)                             
interest_rate  0.0062                                                                             -0.1593**                                                                         
               (0.0082)                                                                       

In [5]:
model2_all_s1 = sm.OLS.from_formula(
    "delta_r ~ r_lag1",
    data = df_cleaned
).fit()

model2_usa_s1 = sm.OLS.from_formula(
    "delta_r ~ r_lag1",
    data = df_cleaned.query("`Country Code` == 'USA'")
).fit()

df_cleaned["delta_r_usa_hat"] = model2_usa_s1.fittedvalues
df_cleaned["r_usa_hat"] = df_cleaned["delta_r_usa_hat"] + df_cleaned["r_lag1"]
df_cleaned["log_gross_r_usa_hat"] = np.log(1+df_cleaned["r_usa_hat"])

df_cleaned["delta_r_all_hat"] = model2_all_s1.fittedvalues
df_cleaned["r_all_hat"] = df_cleaned["delta_r_all_hat"] + df_cleaned["r_lag1"]
df_cleaned["log_gross_r_all_hat"] = np.log(1+df_cleaned["r_all_hat"])

model2_all_bm_s2 = sm.OLS.from_formula(
    "c_growth ~ r_all_hat",
    data = df_cleaned
).fit()

model2_usa_bm_s2 = sm.OLS.from_formula(
    "c_growth ~ r_usa_hat",
    data = df_cleaned.query("`Country Code` == 'USA'")
).fit()

model2_all_logr_s2 = sm.OLS.from_formula(
    "c_growth ~ log_gross_r_all_hat",
    data = df_cleaned
).fit()

model2_usa_logr_s2 = sm.OLS.from_formula(
    "c_growth ~ log_gross_r_usa_hat",
    data = df_cleaned.query("`Country Code` == 'USA'")
).fit()

model2_all_logr_logc_s2 = sm.OLS.from_formula(
    "log_gross_c_growth ~ log_gross_r_all_hat",
    data = df_cleaned
).fit()

model2_usa_logr_logc_s2 = sm.OLS.from_formula(
    "log_gross_c_growth ~ log_gross_r_usa_hat",
    data = df_cleaned.query("`Country Code` == 'USA'")
).fit()

results_table_m2 = summary_col(
    [model2_all_s1, model2_all_bm_s2, model2_all_logr_s2, model2_all_logr_logc_s2],
    stars=True,
    model_names=[ "Stage 1 (all)", "Model 2 (all, BM)", "Model 2 (all, with log(r_hat))", "Model 2 (all, with log(r_hat) and log(c))"],
    info_dict={
        "N": lambda x: f"{int(x.nobs)}"
    },
)
results_table_m2_usa = summary_col(
    [model2_usa_s1, model2_usa_bm_s2, model2_usa_logr_s2, model2_usa_logr_logc_s2],
    stars=True,
    model_names=[ "Stage 1 (US)", "Model 2 (US, BM)", "Model 2 (US, with log(r_hat))", "Model 2 (US, with log(r_hat) and log(c))"],
    info_dict={
        "N": lambda x: f"{int(x.nobs)}"
    },
)
print(results_table_m2)
print(results_table_m2_usa)


                    Stage 1 (all) Model 2 (all, BM) Model 2 (all, with log(r_hat)) Model 2 (all, with log(r_hat) and log(c))
----------------------------------------------------------------------------------------------------------------------------
Intercept           0.0400***     0.0403***         0.0389***                      0.0363***                                
                    (0.0021)      (0.0016)          (0.0017)                       (0.0017)                                 
r_lag1              -0.7287***                                                                                              
                    (0.0121)                                                                                                
r_all_hat                         -0.0173                                                                                   
                                  (0.0222)                                                                                  